<a href="https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)



## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I use regularized logistic regression because the outcome is binary and its predicted probability can rank a review queue. It is a deliberately simple first learned model: it is stable and inspectable, and it earns extra complexity only if it improves held-out Precision@K. Numeric inputs are median-imputed and standardized; categorical inputs are one-hot encoded using training categories only. Missingness flags preserve systematic absence rather than silently treating it as zero.

IDs, the label and its sources, all 30-day trend components, and generation-provider fields are excluded. The remaining 90-day aggregates may describe the same observation period, so this is not forward-looking prediction.

In [2]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import json
import os, sys, subprocess

SEED = 42
TOP_KS = (50, 100)

def show(table):
    print(table.to_string(index=False) if isinstance(table, pd.DataFrame) else table)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Define paths using Path objects, relative to the current working directory (REPO_DIR)
DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
OUTPUT_DIR = Path('work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
assert df['content_id'].is_unique
print(f'Rows loaded: {len(df):,}; one row per pseudonymized content item.')
assert df['content_id'].is_unique
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
for c in ('impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d'):
    df[f'log_{c}'] = np.log1p(df[c].clip(lower=0))

NUMERIC = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
EXCLUDED = {'content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'provider_used', 'model_used'}
assert set(NUMERIC + CATEGORICAL).isdisjoint(EXCLUDED)
FLAGS = []
for c in NUMERIC:
    flag = f'{c}_missing'
    df[flag] = df[c].isna().astype(int)
    FLAGS.append(flag)
FEATURES = NUMERIC + FLAGS + CATEGORICAL
X, y, groups = df[FEATURES].copy(), df['is_declining_label'].copy(), df['client_id'].copy()

def make_matrices(train, test):
    train_parts, test_parts, sources = [], [], []
    for c in NUMERIC + FLAGS:
        a = pd.to_numeric(train[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        b = pd.to_numeric(test[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        median = a.median()
        median = 0.0 if pd.isna(median) else float(median)
        a, b = a.fillna(median).to_numpy(float), b.fillna(median).to_numpy(float)
        mean, std = float(a.mean()), float(a.std())
        std = 1.0 if std == 0 or not np.isfinite(std) else std
        train_parts.append(((a - mean) / std)[:, None])
        test_parts.append(((b - mean) / std)[:, None])
        sources.append(c)
    for c in CATEGORICAL:
        a, b = train[c].fillna('unknown').astype(str), test[c].fillna('unknown').astype(str)
        for value in sorted(a.unique()):
            train_parts.append(a.eq(value).to_numpy(float)[:, None])
            test_parts.append(b.eq(value).to_numpy(float)[:, None])
            sources.append(c)
    return np.hstack(train_parts), np.hstack(test_parts), np.array(sources)

def sigmoid(values):
    return 1 / (1 + np.exp(-np.clip(values, -35, 35)))

def fit_logistic(matrix, target, lr=0.15, l2=0.05, maximum=3000):
    target = np.asarray(target, float)
    weights = np.zeros(matrix.shape[1])
    intercept = float(np.log(target.mean() / (1 - target.mean())))
    for iteration in range(1, maximum + 1):
        residual = sigmoid(matrix @ weights + intercept) - target
        new_weights = weights - lr * ((matrix.T @ residual) / len(target) + l2 * weights)
        new_intercept = intercept - lr * residual.mean()
        if max(np.max(np.abs(new_weights - weights)), abs(new_intercept - intercept)) < 1e-7:
            return new_weights, new_intercept, iteration
        weights, intercept = new_weights, new_intercept
    return weights, intercept, maximum

def probability(matrix, weights, intercept):
    return sigmoid(matrix @ weights + intercept)

def average_precision(actual, scores):
    ordered = np.asarray(actual)[np.argsort(-np.asarray(scores), kind='stable')]
    positives = ordered.sum()
    precision = np.cumsum(ordered) / np.arange(1, len(ordered) + 1)
    return float(precision[ordered == 1].sum() / positives) if positives else 0.0

def roc_auc(actual, scores):
    actual = np.asarray(actual)
    positives, negatives = actual.sum(), len(actual) - actual.sum()
    ranks = pd.Series(scores).rank(method='average').to_numpy()
    return float((ranks[actual == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))

def metrics(actual, scores, rate):
    ordered = np.asarray(actual)[np.argsort(-np.asarray(scores), kind='stable')]
    result = {'average_precision': average_precision(actual, scores), 'roc_auc': roc_auc(actual, scores)}
    for k in TOP_KS:
        p = float(ordered[:k].mean())
        result.update({f'precision_at_{k}': p, f'positives_at_{k}': int(ordered[:k].sum()), f'lift_at_{k}': p / rate})
    return result

def week4_score(frame):
    eligible = frame['impressions_90d'].ge(3000) & frame['avg_position'].between(4, 10) & frame['ctr'].le(0.30)
    return pd.Series(np.where(eligible, frame['impressions_90d'] * (0.30 - frame['ctr']) / 100, 0.0), index=frame.index)

print(f'Rows: {len(df):,}; clients: {groups.nunique()}; observed decline rate: {y.mean():.1%}.')
print(f'Features: {len(FEATURES)}; seed: {SEED}; pandas {pd.__version__}; NumPy {np.__version__}; Python {platform.python_version()}.')

Rows loaded: 30,000; one row per pseudonymized content item.
Rows: 30,000; clients: 32; observed decline rate: 54.2%.
Features: 44; seed: 42; pandas 2.2.2; NumPy 2.0.2; Python 3.12.13.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


A random row split would put pages from the same client into both partitions and exaggerate generalization. I use one fixed random group holdout: 8 of 32 whole pseudonymized clients are test data. The Week-4 rule and the model see exactly the same held-out rows. This tests transfer to unseen clients, not future performance; a one-snapshot dataset has no earlier feature window paired with a later outcome.

In [3]:
rng = np.random.default_rng(SEED)
test_clients = set(rng.choice(np.array(sorted(groups.unique())), size=8, replace=False))
test_mask = groups.isin(test_clients)
X_train, X_test = X.loc[~test_mask], X.loc[test_mask]
y_train, y_test = y.loc[~test_mask], y.loc[test_mask]
assert set(groups.loc[~test_mask]).isdisjoint(set(groups.loc[test_mask]))
split = pd.DataFrame({'partition': ['train', 'held-out test'], 'rows': [len(X_train), len(X_test)], 'clients': [groups.loc[~test_mask].nunique(), groups.loc[test_mask].nunique()], 'observed_decline_rate': [f'{y_train.mean():.1%}', f'{y_test.mean():.1%}']})
show(split)
print('Group check: PASS — no pseudonymized client crosses the split boundary.')

    partition  rows  clients observed_decline_rate
        train 15248       24                 56.5%
held-out test 14752        8                 51.8%
Group check: PASS — no pseudonymized client crosses the split boundary.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Week-4 score is unchanged: at least 3,000 impressions, position 4–10, CTR at or below 0.30%, ranked by its estimated click gap to 0.30% CTR. It is not retrained or threshold-tuned. Both approaches rank only held-out clients. Precision@K is the fraction of the first K candidates with the observed decline label; lift divides it by the held-out base rate. Average precision and ROC AUC are secondary all-threshold checks. Complexity earns a place only if held-out Precision@K improves.

In [4]:
train_matrix, test_matrix, encoded_sources = make_matrices(X_train, X_test)
weights, intercept, iterations = fit_logistic(train_matrix, y_train)
model_scores = pd.Series(probability(test_matrix, weights, intercept), index=X_test.index)
baseline_scores = week4_score(df.loc[X_test.index])
rate = y_test.mean()
comparison = pd.DataFrame([{'approach': 'Week-4 CTR-opportunity rule', **metrics(y_test, baseline_scores, rate)}, {'approach': 'Regularized logistic regression', **metrics(y_test, model_scores, rate)}])
comparison = comparison[['approach', 'precision_at_50', 'positives_at_50', 'lift_at_50', 'precision_at_100', 'positives_at_100', 'lift_at_100', 'average_precision', 'roc_auc']].round(3)
print(f'Logistic regression converged in {iterations:,} batch-gradient iterations with L2=0.05.')
print(f'Held-out observed decline base rate: {rate:.1%}.')
show(comparison)
if comparison.loc[1, 'precision_at_50'] > comparison.loc[0, 'precision_at_50']:
    print('Decision: the logistic model earns a trial for observed-label ranking at K=50.')
else:
    print('Decision: keep the transparent Week-4 rule; logistic regression did not improve held-out Precision@50.')
print('Scope: this measures observed-snapshot alignment, not refresh impact or future ranking change.')

Logistic regression converged in 3,000 batch-gradient iterations with L2=0.05.
Held-out observed decline base rate: 51.8%.
                       approach  precision_at_50  positives_at_50  lift_at_50  precision_at_100  positives_at_100  lift_at_100  average_precision  roc_auc
    Week-4 CTR-opportunity rule             0.50               25       0.965              0.43                43        0.830              0.526    0.509
Regularized logistic regression             0.82               41       1.583              0.78                78        1.506              0.645    0.658
Decision: the logistic model earns a trial for observed-label ranking at K=50.
Scope: this measures observed-snapshot alignment, not refresh impact or future ranking change.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


Permutation importance measures the held-out average-precision drop after all encoded columns belonging to one input are shuffled together. It is association, not causation: correlated traffic features can share or mask importance. I inspect broad error patterns and three redacted wrong cases. IDs, URLs, client names, and raw queries are never displayed. The 0.50 cutoff below describes error types only; it does not choose the ranking queue.

In [5]:
base_ap = average_precision(y_test, model_scores)
perm_rng = np.random.default_rng(SEED)
rows = []
for feature in FEATURES:
    cols = np.flatnonzero(encoded_sources == feature)
    drops = []
    for repeat in range(4):
        shuffled = test_matrix.copy()
        order = perm_rng.permutation(len(shuffled))
        shuffled[:, cols] = shuffled[order][:, cols]
        drops.append(base_ap - average_precision(y_test, probability(shuffled, weights, intercept)))
    rows.append({'feature': feature, 'mean_ap_drop_when_shuffled': np.mean(drops), 'std': np.std(drops)})
importance = pd.DataFrame(rows).sort_values('mean_ap_drop_when_shuffled', ascending=False).head(8).round(4)
print('Top permutation features on held-out clients:')
show(importance)
print('Interpretation: high importance is directional evidence that the fitted rank uses the feature; it does not show that changing the feature would change the outcome.')

errors = df.loc[X_test.index, ['content_type', 'main_intent', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days']].copy()
errors['observed_decline'] = y_test
errors['model_probability'] = model_scores
predicted = model_scores.ge(0.50).astype(int)
errors['error_type'] = np.select([(predicted.eq(1) & y_test.eq(0)), (predicted.eq(0) & y_test.eq(1))], ['false positive', 'false negative'], default='correct')
by_type = errors.loc[errors['error_type'].ne('correct')].groupby(['error_type', 'content_type'], dropna=False).size().rename('n_errors').reset_index().sort_values(['error_type', 'n_errors'], ascending=[True, False])
print('Error counts by content type:')
show(by_type.head(10))

cases = errors.loc[errors['error_type'].ne('correct'), ['error_type', 'model_probability', 'content_type', 'main_intent', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days']].sort_values(['error_type', 'model_probability'], ascending=[True, False]).head(3).reset_index(drop=True)
cases.insert(0, 'redacted_case', [f'case_{i + 1}' for i in range(len(cases))])
cases['why_hard'] = np.where(cases['error_type'].eq('false positive'), 'Allowed snapshot signals resemble declines, but final-30d direction is not down.', 'Allowed snapshot aggregates resemble non-declines, but final-30d direction is down.')
print('Three concrete wrong cases, identifiers intentionally removed:')
show(cases.round({'model_probability': 3, 'ctr': 2, 'avg_position': 1}))
print(f'At descriptive threshold 0.50: {(errors.error_type == 'false positive').sum():,} false positives; {(errors.error_type == 'false negative').sum():,} false negatives.')
print('The comparison table—not this cutoff—controls the review-first decision.')

Top permutation features on held-out clients:
               feature  mean_ap_drop_when_shuffled    std
      content_age_days                      0.0417 0.0028
   log_impressions_90d                      0.0389 0.0014
        log_clicks_90d                      0.0379 0.0016
 days_with_impressions                      0.0177 0.0018
    days_with_sessions                      0.0144 0.0013
          avg_position                      0.0129 0.0010
      log_sessions_90d                      0.0049 0.0024
days_since_last_update                      0.0044 0.0007
Interpretation: high importance is directional evidence that the fitted rank uses the feature; it does not show that changing the feature would change the outcome.
Error counts by content type:
    error_type       content_type  n_errors
false negative    keyword article       788
false negative comparison article         2
false positive    keyword article      4850
false positive comparison article       295
Three concrete wro

## Self-check


- [x] All required sections contain reasoning and runnable code.
- [x] Logistic regression fits the observed binary label; probabilities rank the review queue.
- [x] The fixed grouped holdout keeps every client entirely in train or test.
- [x] The unchanged Week-4 rule and model use the same held-out rows, base rate, and Precision@50/100 metrics.
- [x] IDs, raw queries, trend fields, all 30-day trend components, and provider/model fields are excluded.
- [x] Missingness flags are added before median imputation; no blind fillna(0) is used.
- [x] Feature interpretation, aggregate errors, and three redacted wrong cases are shown.
- [x] Claims are observed, directional, and decision-support only.
- [x] I read the local *State of AI-Driven SEO* research report before this task.
- [x] Notebook executed top to bottom in the repository environment.